In [36]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.metrics import r2_score
import os
os.chdir('E:/Machine_Learning/Datasets')

In [37]:
df = pd.read_csv("ChemicalProcess.csv")
df

,Yield,BiologicalMaterial01,BiologicalMaterial02,BiologicalMaterial03,BiologicalMaterial04,BiologicalMaterial05,BiologicalMaterial06,BiologicalMaterial07,BiologicalMaterial08,BiologicalMaterial09,...,ManufacturingProcess36,ManufacturingProcess37,ManufacturingProcess38,ManufacturingProcess39,ManufacturingProcess40,ManufacturingProcess41,ManufacturingProcess42,ManufacturingProcess43,ManufacturingProcess44,ManufacturingProcess45
0,38.00,6.25,49.58,56.97,12.74,19.51,43.73,100.0,16.66,11.44,...,0.019,0.5,3,7.2,NaN,NaN,11.6,3.0,1.8,2.4
1,42.44,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.019,2.0,2,7.2,0.1,0.15,11.1,0.9,1.9,2.2
2,42.03,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.018,0.7,2,7.2,0.0,0.00,12.0,1.0,1.8,2.3
3,41.42,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.018,1.2,2,7.2,0.0,0.00,10.6,1.1,1.8,2.1
4,42.49,7.47,63.33,72.25,14.02,17.91,54.66,100.0,18.22,12.80,...,0.017,0.2,2,7.3,0.0,0.00,11.0,1.1,1.7,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,39.66,6.71,56.32,66.19,12.35,20.02,50.26,100.0,17.54,12.50,...,NaN,2.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
172,39.68,6.87,56.74,66.61,12.55,20.18,50.80,100.0,17.48,12.41,...,NaN,1.0,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
173,42.23,7.50,58.41,68.30,13.33,20.81,52.96,100.0,17.23,12.04,...,NaN,1.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
174,38.48,7.53,58.36,69.25,14.35,20.57,51.31,100.0,17.87,12.77,...,NaN,2.3,0,0.0,0.0,0.00,0.0,0.5,0.0,0.0


In [38]:
df.isna().sum().sum()

np.int64(106)

In [39]:
X = df.drop('Yield',axis = 1)
y = df['Yield']

## Imputation must be in flow => split the data => fit_transform() on X_train , transform() on X_test

In [41]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26)


In [43]:
im = SimpleImputer(strategy = 'mean').set_output(transform='pandas')
X_train_imp = im.fit_transform(X_train)
X_test_imp = im.transform(X_test)

In [44]:
model = LinearRegression()
model.fit(X_train_imp,y_train)
y_pred = model.predict(X_test_imp)

In [45]:
r2 = r2_score(y_test,y_pred)
r2

-7.299578219035819

#### R2 score can be negative sometimes (worse case)

# Try Ridge, Lasso, ElasticNet

In [57]:
alphas = np.linspace(0.001,10,20)
scores = []
for a in alphas:
    ridge = Ridge(alpha = a)
    ridge.fit(X_train_imp,y_train)
    y_pred = ridge.predict(X_test_imp)
    r2 = r2_score(y_test,y_pred)
    scores.append([a,r2])

df_r2_scores_ridge = pd.DataFrame(scores,columns=['alpha','r2 score'])
df_r2_scores_ridge.nlargest(1,'r2 score')

,alpha,r2 score
4,2.106053,-2.459765


In [58]:
alphas = np.linspace(0.001,10,20)
scores = []
for a in alphas:
    lasso = Lasso(alpha = a)
    lasso.fit(X_train_imp,y_train)
    y_pred = lasso.predict(X_test_imp)
    r2 = r2_score(y_test,y_pred)
    scores.append([a,r2])

df_r2_scores_ridge = pd.DataFrame(scores,columns=['alpha','r2 score'])
df_r2_scores_ridge.nlargest(1,'r2 score')

C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.035920e+01, tolerance: 4.417e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.011577e+01, tolerance: 4.417e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to inc

,alpha,r2 score
15,7.894947,0.325685


In [59]:
alphas = np.linspace(0.001,10,20)
ratios = np.linspace(0.001,1,20)
scores = []
for a in alphas:
    for r in ratios:
        el = ElasticNet(alpha=a,l1_ratio=r)
        el.fit(X_train_imp,y_train)
        y_pred = el.predict(X_test_imp)
        r2 = r2_score(y_test,y_pred)
        scores.append([a,r,r2])

df_r2_scores_elastic = pd.DataFrame(scores,columns=['alpha','l1 ratio','r2 score'])
df_r2_scores_elastic.nlargest(1,'r2 score')

C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.933718e+01, tolerance: 4.417e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.939665e+01, tolerance: 4.417e-02
  model = cd_fast.enet_coordinate_descent(
C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to inc

,alpha,l1 ratio,r2 score
27,0.527263,0.369053,0.577195
